In [4]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [15]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported successfully.")
print("Ready to build a RAG system")

All libraries imported successfully.
Ready to build a RAG system


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)
print("api key initialized")

api key initialized


In [17]:
df = pd.read_csv('college_notes.csv')
print("shape of dataset:" , df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nfirst 3 rows:")
print(df.head(3))
print(df.tail(3))

shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']

first 3 rows:
   note_id           subject             topic  \
0        1  Data Engineering     ETL Pipelines   
1        2  Data Engineering  Data Warehousing   
2        3  Data Engineering      Apache Spark   

                                             content  
0  ETL stands for Extract, Transform, Load. It is...  
1  A data warehouse is a central repository that ...  
2  Apache Spark is an open-source distributed com...  
    note_id subject            topic  \
12       13   GenAI      RAG Systems   
13       14  Python   Pandas Library   
14       15  Python  API Integration   

                                              content  
12  Retrieval-Augmented Generation or RAG is an AI...  
13  Pandas is a Python library for data manipulati...  
14  An API or Application Programming Interface al...  


In [18]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())
print("\nSamples of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content (number df characters) for each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

Subjects in the dataset:
subject
Data Engineering    5
GenAI               5
Machine Learning    3
Python              2
Name: count, dtype: int64

Samples of topics:
 note_id          subject                  topic
       1 Data Engineering          ETL Pipelines
       2 Data Engineering       Data Warehousing
       3 Data Engineering           Apache Spark
       4 Data Engineering Medallion Architecture
       5 Data Engineering         Data Pipelines
       6 Machine Learning      Linear Regression
       7 Machine Learning    Feature Engineering
       8 Machine Learning       Model Evaluation
       9            GenAI  Large Language Models
      10            GenAI     Prompt Engineering
      11            GenAI             Embeddings
      12            GenAI       Vector Databases
      13            GenAI            RAG Systems
      14           Python         Pandas Library
      15           Python        API Integration

Length of content (number df characters) for eac

In [19]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared: {len(documents)}")
print(f"First documents ID:{ids[0]}")
print(f"First metadata:{metadatas[0]}")
print(f"First 100 chars of doc:{documents[0][:100]}...")

Total chunks prepared: 15
First documents ID:note_1
First metadata:{'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc:ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from ...


In [20]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded successfully")
test_embedding = embedding_model.encode("this is a test sentence")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"Test embedding: {test_embedding[:5]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully
Test embedding shape: (384,)
Test embedding: [0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [21]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name = "college_notes_rag")
print("chromaDB client created.")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection: {collection.count()}")


chromaDB client created.
Collection name: college_notes_rag
Documents in collection: 0


In [22]:
embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"Embeddings matrix shape: {embeddings.shape}")
embedding_list = embeddings.tolist()
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings
)
print(f"\nDocuments successfully added")
print(f"Total documents in collection: {collection.count()}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings matrix shape: (15, 384)

Documents successfully added
Total documents in collection: 15


In [23]:
def retrive_relavant_chunks(test_question, top_k=3):
  question_embedding = embedding_model.encode(test_question).tolist()
  results = collection.query(
    query_embeddings=[question_embedding],
    n_results=top_k
)
  return results
print("retrived")


retrived


In [24]:
test_question = "what is Linear Regression?"
print(f"Test Question: {test_question}")
results = retrive_relavant_chunks(test_question, top_k=3)
print("\nTop 3 Retrived chunks:")
for i, (doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
 print(f"\nResults {i+1}:")
 print(f"Subject : {meta['subject']}")
 print(f"Topic : {meta['topic']}")
 print(f"Distance : {dist:.4f}")
 print(f"Content : {doc[:120]}...")


Test Question: what is Linear Regression?

Top 3 Retrived chunks:

Results 1:
Subject : Machine Learning
Topic : Linear Regression
Distance : 0.3008
Content : Linear regression is a supervised machine learning algorithm used to predict a continuous numerical value. It finds the ...

Results 2:
Subject : Machine Learning
Topic : Model Evaluation
Distance : 1.4306
Content : Model evaluation measures how well a machine learning model performs on unseen data. For regression models, common metri...

Results 3:
Subject : Data Engineering
Topic : ETL Pipelines
Distance : 1.5318
Content : ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from source systems into ...


In [25]:
def build_context_from_results(results):
  """
  Format Chromadb retrival results into a readable string
  """
  context_parts = []
  for i, (doc,meta) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0]
  )):
    chunk_text = f"[Source {i+1} : {meta['subject']} - {meta['topic']}]\n{doc}"
    context_parts.append(chunk_text)
  context_str = "\n\n---\n\n".join(context_parts)
  return context_str
context = build_context_from_results(results)
print("built context string ")


built context string 


In [26]:
def generate_rag_answer(question, context):
  system_prompt = """
    Answer ONLY using the information provided in the context below.
    If the answer is not found in the context, say exactly:
    "I don't have enough information in my knowledge base to answer this question."
    Do not use your general training knowledge.
    Keep answers clear, accurate, and beginner-friendly.
    Mention which source the information came from when possible.
  """

  user_prompt = f"""Context from Knowledge Base:
{context}
Student's Question: {question}
Please answer the question based only on the context provided above."""

  response = groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature=0.1,
      max_tokens=500
  )
  answer = response.choices[0].message.content
  return answer

print("RAG generation function defined.")

RAG generation function defined.


In [27]:
def ask_college_assistant(question, top_k=3, verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("step 1: Retrieving relevant documents...")
  results = retrive_relavant_chunks(question, top_k=top_k)
  if verbose:
    print(f"Retrieved {top_k} chunks from knowlege base:")
    for i,meta in enumerate(results['metadatas'][0]):
      print(f"\nResult {i+1}:")
    print("\nStep 2: Building context string ..")
  context = build_context_from_results(results)
  if verbose:
    print(f"Context build ({len(context)} characters)")
    print("\nStep 3: Sending to LLM for answer generation...")
  answer = generate_rag_answer(question, context)
  if verbose:
    print(answer)
  return answer
print("completed")

completed


In [ ]:
import os
from groq import Groq

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)
print("api key initialized")

api key initialized


In [28]:
question_1 = "what is ELT?"
answer_1 = ask_college_assistant(question_1, top_k=3, verbose=True)


Question: what is ELT?
step 1: Retrieving relevant documents...
Retrieved 3 chunks from knowlege base:

Result 1:

Result 2:

Result 3:

Step 2: Building context string ..
Context build (1673 characters)

Step 3: Sending to LLM for answer generation...
I don't have enough information in my knowledge base to answer this question.
